# Week 1

# 🎙️ Speech Emotion Recognition Using Machine Learning
---
This notebook builds a complete Speech Emotion Recognition (SER) system step by step.

The system listens to audio recordings and detects the emotion of the speaker —
such as happy, sad, angry, neutral, fearful, and more.

We follow a clear pipeline:
1. Load and explore the datasets
2. Clean and prepare the audio files
3. Extract features from each audio file
4. Train and compare four models
5. Evaluate and present the final results

All four datasets are free and available on Kaggle.
All experiments run on Kaggle free GPU — no local setup needed.

## Section 1 — Import Libraries
---
In this section we import all the Python libraries we need for the entire project.

- **librosa** — reads audio files and extracts features like MFCC
- **numpy and pandas** — handle data and organise it into tables
- **scikit-learn** — used for the SVM baseline model and data scaling
- **PyTorch** — used to build CNN, CNN-LSTM and Attention deep learning models
- **matplotlib and seaborn** — used to plot graphs and confusion matrices
- **os and glob** — used to navigate folders and find audio files

In [ ]:
# ─────────────────────────────────────────────
# SECTION 1 — IMPORT LIBRARIES
# ─────────────────────────────────────────────

# Audio processing
import librosa
import librosa.display

# Data handling
import numpy as np
import pandas as pd

# File and folder navigation
import os
import glob

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning utilities
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report,
                             confusion_matrix)

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Progress bar
from tqdm import tqdm

# Suppress warnings for clean output
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("All libraries imported successfully.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# MASTER OUTPUT FOLDER SETUP
# Run this once at the start — creates all section folders
# ─────────────────────────────────────────────────────────────

import os

# Define all section output folders
folders = [
    '/kaggle/working/outputs',
    '/kaggle/working/outputs/section2_datasets',
    '/kaggle/working/outputs/section3_features',
    '/kaggle/working/outputs/section4_svm',
    '/kaggle/working/outputs/section5_cnn',
    '/kaggle/working/outputs/section6_cnn_lstm',
    '/kaggle/working/outputs/section7_attention',
    '/kaggle/working/outputs/section8_results',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("All output folders created successfully.")
print()
for folder in folders:
    print(f"  ✓  {folder}")

## Section 2 — Load Datasets
---
We use four publicly available speech emotion datasets. Each dataset contains
audio recordings of actors speaking with different emotions.

| Dataset | Speakers | Emotions | Files |
|---------|----------|----------|-------|
| RAVDESS | 24 professional actors | 8 emotions | ~1,440 |
| CREMA-D | 91 actors | 6 emotions | 7,442 |
| TESS | 2 female speakers | 7 emotions | 2,800 |
| SAVEE | 4 male speakers | 7 emotions | 480 |

All datasets are already added to this notebook via Kaggle.
We load all audio file paths and their emotion labels into one combined dataframe.

The emotion labels across all datasets are mapped to these common categories:
angry, disgust, fear, happy, neutral, sad, surprised, calm.

In [ ]:
# ── Label extractor functions ──

import os
import glob
import pandas as pd

def get_ravdess_label(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('-')
    if len(parts) < 3:
        return None
    try:
        emotion_code = int(parts[2])
    except ValueError:
        return None
    emotion_map = {
        1: 'neutral', 2: 'calm',     3: 'happy', 4: 'sad',
        5: 'angry',   6: 'fearful',  7: 'disgust', 8: 'surprised'
    }
    return emotion_map.get(emotion_code, None)

def get_cremad_label(filepath):
    filename = os.path.basename(filepath)
    parts = filename.split('_')
    if len(parts) < 3:
        return None
    emotion_code = parts[2]
    emotion_map = {
        'ANG': 'angry',   'DIS': 'disgust', 'FEA': 'fearful',
        'HAP': 'happy',   'NEU': 'neutral', 'SAD': 'sad'
    }
    return emotion_map.get(emotion_code, None)

def get_tess_label(filepath):
    folder = os.path.basename(os.path.dirname(filepath)).lower()
    emotion_map = {
        'angry':   'angry',   'disgust': 'disgust',
        'fear':    'fearful', 'happy':   'happy',
        'neutral': 'neutral', 'sad':     'sad',
        'ps':      'surprised'
    }
    for key in emotion_map:
        if key in folder:
            return emotion_map[key]
    return None

def get_savee_label(filepath):
    filename = os.path.basename(filepath).lower()
    parts = filename.split('_')
    if len(parts) < 2:
        return None
    code = parts[1][:2].strip()
    emotion_map = {
        'a':  'angry',   'd':  'disgust', 'f':  'fearful',
        'h':  'happy',   'n':  'neutral', 'sa': 'sad',
        'su': 'surprised'
    }
    return emotion_map.get(code, emotion_map.get(code[0], None))

print("Label functions defined successfully.")

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2 — LOAD DATASETS (FIXED PATHS)
# ─────────────────────────────────────────────

def load_all_datasets():
    data = []

    # ── RAVDESS ──
    ravdess_files = glob.glob(
        '/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio/**/*.wav',
        recursive=True
    )
    for f in ravdess_files:
        label = get_ravdess_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'RAVDESS'})

    # ── CREMA-D ──
    # CREMA-D is inside the dmitrybabko combined pack
    cremad_files = glob.glob(
        '/kaggle/input/datasets/dmitrybabko/speech-emotion-recognition-en/Crema/**/*.wav',
        recursive=True
    )
    for f in cremad_files:
        label = get_cremad_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'CREMAD'})

    # ── TESS ──
    tess_files = glob.glob(
        '/kaggle/input/datasets/ejlok1/toronto-emotional-speech-set-tess/**/*.wav',
        recursive=True
    )
    for f in tess_files:
        label = get_tess_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'TESS'})

    # ── SAVEE ──
    savee_files = glob.glob(
        '/kaggle/input/datasets/dmitrybabko/speech-emotion-recognition-en/Savee/**/*.wav',
        recursive=True
    )
    for f in savee_files:
        label = get_savee_label(f)
        if label:
            data.append({'path': f, 'emotion': label, 'source': 'SAVEE'})

    return pd.DataFrame(data)


# Load everything
df = load_all_datasets()

print(f"Total audio files loaded: {len(df)}")
print(f"\nFiles per dataset:")
print(df['source'].value_counts())
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())

In [ ]:
# ── 2.3 Visualise emotion distribution ──

plt.figure(figsize=(12, 4))

# Emotion count
plt.subplot(1, 2, 1)
df['emotion'].value_counts().plot(kind='bar', color='#2e7d32')
plt.title('Emotion Distribution (All Datasets)')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)

# Dataset count
plt.subplot(1, 2, 2)
df['source'].value_counts().plot(kind='bar', color='#1b5e20')
plt.title('Files per Dataset')
plt.xlabel('Dataset')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# ── Save Section 2 Outputs ──

import matplotlib.pyplot as plt
import seaborn as sns

# Plot 1 — Emotion distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['emotion'].value_counts().plot(
    kind='bar', color='#2e7d32', ax=axes[0]
)
axes[0].set_title('Emotion Distribution (All Datasets)')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

df['source'].value_counts().plot(
    kind='bar', color='#1b5e20', ax=axes[1]
)
axes[1].set_title('Files Per Dataset')
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(
    '/kaggle/working/outputs/section2_datasets/dataset_distribution.png',
    dpi=150
)
plt.close()

# Save dataset summary CSV
df.groupby(['source', 'emotion']).size()\
  .reset_index(name='count')\
  .to_csv(
      '/kaggle/working/outputs/section2_datasets/dataset_summary.csv',
      index=False
  )

# Save JSON summary
import json
summary_s2 = {
    'total_files'        : int(len(df)),
    'files_per_dataset'  : df['source'].value_counts().to_dict(),
    'emotion_distribution': df['emotion'].value_counts().to_dict()
}
with open('/kaggle/working/outputs/section2_datasets/dataset_info.json', 'w') as f:
    json.dump(summary_s2, f, indent=4)

print("Section 2 outputs saved.")
print("  dataset_distribution.png")
print("  dataset_summary.csv")
print("  dataset_info.json")

## week 2


## Section 3 — Data Preprocessing
---
Raw audio files cannot be used directly for training. We need to clean and
prepare them first. This section does three things:

### 3.1 — Silence Trimming
Each audio file may have silence at the beginning or end.
We remove this silence so the model only learns from actual speech.

### 3.2 — Amplitude Normalisation
Different recordings have different volume levels.
We normalise all files to the same volume so the model is not confused
by loud or quiet recordings.

### 3.3 — Data Augmentation
Our combined dataset has around 12,000 audio files.
To make the model more robust and reduce overfitting, we create extra
training samples by slightly modifying existing ones:

- **Gaussian Noise** — adds very small random noise to the audio
- **Pitch Shifting** — shifts the pitch up or down slightly (±2 semitones)
- **Time Stretching** — speeds up or slows down the audio slightly (rate 0.8–1.2)

This effectively doubles or triples our training data without collecting new recordings.

### 3.4 — Label Encoding
Emotion labels are text (e.g. "happy", "sad").
We convert them to numbers (e.g. 0, 1, 2...) so the model can process them.

After this section, we have a clean, augmented, labelled set of audio files
ready for feature extraction in Section 4.

In [ ]:
# # ─────────────────────────────────────────────
# # SECTION 3 — PREPROCESSING + FEATURE EXTRACTION
# # (Memory Safe Version — No raw audio stored)
# # ─────────────────────────────────────────────

# import librosa
# import numpy as np
# from tqdm import tqdm

# SR = 22050
# DURATION = 3.0
# TARGET_LENGTH = int(SR * DURATION)
# N_MFCC = 40

# def extract_features(audio, sr):
#     """Extract all features from one audio array"""

#     # MFCC — 40 coefficients
#     mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
#     mfcc_mean = np.mean(mfcc, axis=1)
#     mfcc_std  = np.std(mfcc,  axis=1)

#     # Mel-spectrogram
#     mel = librosa.feature.melspectrogram(y=audio, sr=sr)
#     mel_mean = np.mean(mel, axis=1)

#     # Chroma
#     chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
#     chroma_mean = np.mean(chroma, axis=1)

#     # Zero Crossing Rate
#     zcr = librosa.feature.zero_crossing_rate(audio)
#     zcr_mean = np.mean(zcr)

#     # Spectral Contrast
#     contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
#     contrast_mean = np.mean(contrast, axis=1)

#     # Tonnetz
#     harmonic = librosa.effects.harmonic(audio)
#     tonnetz = librosa.feature.tonnetz(y=harmonic, sr=sr)
#     tonnetz_mean = np.mean(tonnetz, axis=1)

#     # Combine all into one feature vector
#     features = np.concatenate([
#         mfcc_mean, mfcc_std,
#         mel_mean,
#         chroma_mean,
#         [zcr_mean],
#         contrast_mean,
#         tonnetz_mean
#     ])

#     return features


# def load_clean_extract(filepath, sr=SR, duration=DURATION):
#     """Load one file, clean it, extract features"""
#     try:
#         audio, sr_ = librosa.load(filepath, sr=sr, duration=duration)

#         # Trim silence
#         audio, _ = librosa.effects.trim(audio, top_db=20)

#         # Fix length
#         if len(audio) < TARGET_LENGTH:
#             audio = np.pad(audio, (0, TARGET_LENGTH - len(audio)))
#         else:
#             audio = audio[:TARGET_LENGTH]

#         # Normalise
#         if np.max(np.abs(audio)) > 0:
#             audio = audio / np.max(np.abs(audio))

#         return audio, sr_

#     except Exception:
#         return None, None


# def augment_and_extract(filepath, label):
#     """
#     Load one file, augment it 3 ways,
#     extract features from original + 3 augmented versions.
#     Returns list of (feature_vector, label)
#     """
#     results = []

#     audio, sr = load_clean_extract(filepath)
#     if audio is None:
#         return results

#     versions = []

#     # Original
#     versions.append(audio)

#     # Augmentation 1 — Gaussian noise
#     noisy = audio + 0.005 * np.random.randn(len(audio))
#     versions.append(noisy)

#     # Augmentation 2 — Pitch shift
#     try:
#         pitched = librosa.effects.pitch_shift(audio, sr=sr, n_steps=2)
#         pitched = pitched[:TARGET_LENGTH] if len(pitched) > TARGET_LENGTH else np.pad(pitched, (0, TARGET_LENGTH - len(pitched)))
#         versions.append(pitched)
#     except Exception:
#         versions.append(audio)

#     # Augmentation 3 — Time stretch
#     try:
#         stretched = librosa.effects.time_stretch(audio, rate=0.9)
#         stretched = stretched[:TARGET_LENGTH] if len(stretched) > TARGET_LENGTH else np.pad(stretched, (0, TARGET_LENGTH - len(stretched)))
#         versions.append(stretched)
#     except Exception:
#         versions.append(audio)

#     # Extract features from each version
#     for v in versions:
#         feat = extract_features(v, sr)
#         results.append((feat, label))

#     return results


# # ── Build full feature dataset ──

# print("Extracting features from all files (with augmentation)...")
# print("This will take 15-20 minutes. Please wait.\n")

# all_features = []
# all_labels   = []

# for _, row in tqdm(df.iterrows(), total=len(df)):
#     results = augment_and_extract(row['path'], row['emotion'])
#     for feat, label in results:
#         all_features.append(feat)
#         all_labels.append(label)

# # Convert to numpy arrays
# X = np.array(all_features)
# y = np.array(all_labels)

# print(f"\nFeature matrix shape:  {X.shape}")
# print(f"Labels shape:          {y.shape}")
# print(f"Unique emotions:       {np.unique(y)}")

In [ ]:
# ── Load saved features ──
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

X_scaled  = np.load('/kaggle/input/datasets/zavidd/ser-data/X_scaled.npy')
y_encoded = np.load('/kaggle/input/datasets/zavidd/ser-data/y_encoded.npy')
y         = np.load('/kaggle/input/datasets/zavidd/ser-data/y_labels.npy', allow_pickle=True)

# Rebuild label encoder
le = LabelEncoder()
le.fit(y)

print("Data loaded successfully.")
print(f"X_scaled shape : {X_scaled.shape}")
print(f"y_encoded shape: {y_encoded.shape}")
print(f"Emotions       : {le.classes_}")

In [ ]:
# ── Label Encoding ──
from sklearn.preprocessing import LabelEncoder, StandardScaler

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Label encoding:")
for i, emotion in enumerate(le.classes_):
    print(f"  {emotion:12s} → {i}")

In [ ]:
# ── Save Section 3 Outputs ──

import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt

# Save feature matrix and labels (already saved but save again to output folder)
np.save('/kaggle/working/outputs/section3_features/X_scaled.npy', X_scaled)
np.save('/kaggle/working/outputs/section3_features/y_encoded.npy', y_encoded)
np.save('/kaggle/working/outputs/section3_features/y_labels.npy', y)

# Save label encoding map as CSV
label_map = pd.DataFrame({
    'emotion' : le.classes_,
    'encoded' : list(range(len(le.classes_)))
})
label_map.to_csv(
    '/kaggle/working/outputs/section3_features/label_encoding.csv',
    index=False
)

# Save feature summary as JSON
summary_s3 = {
    'total_samples'      : int(X_scaled.shape[0]),
    'features_per_sample': int(X_scaled.shape[1]),
    'original_files'     : int(len(df)),
    'augmentation_factor': 4,
    'emotions'           : list(le.classes_),
    'feature_components' : {
        'mfcc_mean'       : 40,
        'mfcc_std'        : 40,
        'mel_spectrogram' : 128,
        'chroma'          : 12,
        'zcr'             : 1,
        'spectral_contrast': 7,
        'tonnetz'         : 6
    },
    'total_features': 234
}
with open('/kaggle/working/outputs/section3_features/feature_summary.json', 'w') as f:
    json.dump(summary_s3, f, indent=4)

# Plot emotion distribution after augmentation
import seaborn as sns
unique, counts = np.unique(y, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar(unique, counts, color='#2e7d32')
plt.title('Emotion Distribution After Augmentation')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    '/kaggle/working/outputs/section3_features/augmented_distribution.png',
    dpi=150
)
plt.close()

print("Section 3 outputs saved.")
print("  X_scaled.npy")
print("  y_encoded.npy")
print("  y_labels.npy")
print("  label_encoding.csv")
print("  feature_summary.json")
print("  augmented_distribution.png")